# Experiment D - Mixed Raw + Reconstruction Training

**Question:** can exposure to both raw and reconstructed acceleration during training improve domain robustness and held-out-user generalization?

Strict D2 protocol:

1. Reuse Experiment A's **train / validation / test users** and `class_to_idx` from the A checkpoint.
2. Use the A checkpoint **only for split/class metadata**. Do **not** reuse A model weights or normalization.
3. Fit normalization on **mixed training data**: each training segment contributes both raw and reconstruction valid samples.
4. Train one new `MaskAwareAccelerationCNN` from scratch on the duplicated mixed training set.
5. Select the best epoch using **mixed validation balanced accuracy**.
6. Freeze the selected model and evaluate the **same model** on:
   - raw held-out-user test data;
   - reconstructed held-out-user test data.
7. Use the same mixed-train gallery/prototypes/probe-training data in both representation evaluations.
8. Compute paired raw/reconstruction embedding preservation for the same test samples.

The executable experiment lives in `scripts/run_experiment_d.py`; this notebook is intentionally a thin inspection layer.


In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
from pathlib import Path
import sys

import pandas as pd
import torch


def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root containing the 'snn' directory."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())
SCRIPTS_DIR = REPOSITORY_ROOT / "scripts"

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from snn.accel_reconstruction_eval import experiment_d_config
from run_experiment_d import run_experiment_d

print("Repository root:", REPOSITORY_ROOT)
print("CUDA available:", torch.cuda.is_available())


## Paths and experiment controls

The reference A checkpoint is used only to recover the exact split and class mapping. Keep `ALLOW_NEW_SPLIT=False` for strict A/B/C/D comparability.

Regenerate Experiment A before running B/C/D whenever A's exclusions change. D has no local cohort or explicit-split override.


In [ ]:
DATASET_ROOT = Path(
    "outputs/action0_rectified/low-pass/aligned-board-events"
)
REFERENCE_A_CHECKPOINT = Path(
    "notebooks/artifacts/acceleration_cnn_representation/"
    "best_acceleration_cnn.pt"
)
OUTPUT_DIR = Path(
    "notebooks/artifacts/experiment_D_mixed_training"
)

ALLOW_NEW_SPLIT = False

# Training overrides. Defaults match the current baseline/C configuration.
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 10
BATCH_SIZE = 128
NUM_WORKERS = 0
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.0
USE_CLASS_WEIGHTS = False
GRAD_CLIP_NORM = None
MAX_TRAIN_BATCHES = None
USE_GPU = True


## Build the Experiment D configuration

The nominal config uses `test_source="raw"`, but the D runner always evaluates both raw and reconstruction test domains using the same trained model.


In [ ]:
config = experiment_d_config(
    test_source="raw",
    output_dir=OUTPUT_DIR,
    random_seed=12345,
)

config = replace(
    config,
    use_gpu=USE_GPU,
    loader=replace(
        config.loader,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    ),
    training=replace(
        config.training,
        num_epochs=NUM_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        use_class_weights=USE_CLASS_WEIGHTS,
        grad_clip_norm=GRAD_CLIP_NORM,
        max_train_batches=MAX_TRAIN_BATCHES,
    ),
)
config.validate()
config.to_dict()


## Run Experiment D

This performs:

`load -> recover A split -> fit mixed normalization -> mixed train/val -> select best epoch -> raw test + reconstruction test -> paired preservation -> save artifacts`


In [ ]:
run = run_experiment_d(
    root=DATASET_ROOT,
    repository_root=REPOSITORY_ROOT,
    output_dir=OUTPUT_DIR,
    reference_checkpoint=REFERENCE_A_CHECKPOINT,
    config=config,
    allow_new_split=ALLOW_NEW_SPLIT,
)


## Inherited cohort provenance

Experiment D inherits the cohort from the authoritative A checkpoint.


In [ ]:
provenance = json.loads(
    (run.output_dir / "provenance.json").read_text(encoding="utf-8")
)
display(
    pd.Series(
        {
            "cohort_source": provenance["cohort_source"],
            "excluded_users": provenance["excluded_users"],
            "eligible_users": provenance["eligible_users"],
            "train_users": provenance["train_users"],
            "val_users": provenance["val_users"],
            "test_users": provenance["test_users"],
            "class_to_idx": provenance["class_to_idx"],
        },
        name="Inherited cohort",
    )
)


## Training and split checks


In [ ]:
display(run.split_summary)
display(run.label_split_counts)
display(run.classification_splits)
display(run.training_history.tail(12))
print("Normalization fitted on:", run.normalization.fitted_on)
print("Best checkpoint:", run.checkpoint_path)


## Dual-domain representation results

The two summaries below use the same mixed-trained CNN and the same mixed train/validation reference embeddings. Only the held-out test input domain changes.


In [ ]:
print("D: mixed training -> raw test")
display(run.raw_summary.T.rename(columns={0: "value"}))

print("D: mixed training -> reconstruction test")
display(run.reconstruction_summary.T.rename(columns={0: "value"}))


## Direct raw-vs-reconstruction comparison for the mixed-trained model


In [ ]:
display(run.domain_comparison)

comparison = run.domain_comparison.copy()
if {
    "D_mixed_to_raw",
    "D_mixed_to_reconstruction",
}.issubset(comparison.columns):
    comparison["delta_reconstruction_minus_raw"] = (
        comparison["D_mixed_to_reconstruction"]
        - comparison["D_mixed_to_raw"]
    )
display(comparison)


## Paired preservation

These diagnostics compare the same held-out test samples before and after reconstruction, under the same mixed-trained model.


In [ ]:
display(run.paired_summary)
display(run.paired_transitions)


## Saved artifacts


In [ ]:
artifact_table = pd.DataFrame(
    [
        {"artifact": name, "path": str(path)}
        for name, path in sorted(run.artifact_paths.items())
    ]
)
display(artifact_table)


# Visualization

The cells below are presentation-only. They consume the metrics and embeddings already generated by `run_experiment_d()`.

In [ ]:
from snn.accel_reconstruction_eval.visualization import visualize_experiment_d

figures = visualize_experiment_d(run.output_dir)

## Interpretation checklist

Focus on these comparisons after A/B/C are available:

- **D mixed -> raw vs A raw -> raw:** does mixed training preserve or improve raw-domain generalization?
- **D mixed -> reconstruction vs B raw -> reconstruction:** does mixed training reduce the raw-to-reconstruction domain gap?
- **D mixed -> reconstruction vs C reconstruction -> reconstruction:** does mixed training sacrifice or improve reconstruction-domain specialization?
- **D paired cosine/agreement vs B paired cosine/agreement:** does mixed training make the representation more invariant to reconstruction?

A useful D result should ideally improve robustness to reconstruction without materially degrading raw held-out-user performance.
